# Write a solution to find the second highest distinct salary from the Employee table. If there is no second highest salary, return null (return None in Pandas).

The result format is in the following example.

 

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType

spark = SparkSession.builder.getOrCreate()

# Schema
employee_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("salary", IntegerType(), True),
])

# Example 1 data
employee_data_1 = [
    (1, 100),
    (2, 200),
    (3, 300),
]

employee_df_1 = spark.createDataFrame(employee_data_1, schema=employee_schema)

# Example 2 data
employee_data_2 = [
    (1, 100),
]

employee_df_2 = spark.createDataFrame(employee_data_2, schema=employee_schema)


In [15]:
employee_df_1.show()
employee_df_2.show()

+---+------+
| id|salary|
+---+------+
|  1|   100|
|  2|   200|
|  3|   300|
+---+------+

+---+------+
| id|salary|
+---+------+
|  1|   100|
+---+------+



In [33]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
# performance not good at scale use distinct and order by
window_spec=Window.orderBy(col("salary").desc())
employee_df_1.withColumn("rank",dense_rank().over(window_spec)).where(col("rank")==2).show()

+---+------+----+
| id|salary|rank|
+---+------+----+
|  2|   200|   2|
+---+------+----+



26/01/06 03:37:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 03:37:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 03:37:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 03:37:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 03:37:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [51]:
#fails for 2nd df because spark spark desnot guarantee order after limit(2)
employee_df_1.select("salary").distinct().orderBy("salary",ascending=False).limit(2).orderBy("salary").limit(1).show()

+------+
|salary|
+------+
|   200|
+------+



In [55]:
# works on tb scale
employee_df_1.agg(max("salary")).collect()[0][0]


300